In [ ]:
#| default_exp entities

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
import ast, math, re, sys
from functools import lru_cache
import numpy as np
from fastcore.all import AttrDict, L, Path, chunked, defaults, first, ifnone, merge, patch, store_attr
from fastcore.parallel import ProcessPoolExecutor
from fastlite import Database
from apswutils.db import Table
from multiprocessing import get_context
from litesearch.core import (sql_in, rowid_sel, content_id, NP_DTYPE, process_content, write_txn, db_lock,
                             upsert_all, rrf_all)
from litesearch.topics import get_graph
from litesearch.utils import hash_embed

## The lexical guard

`_lex_ok` decides whether two names are one thing. Embeddings alone merge `python 3.11` into
`python 3.12`, so a merge also needs matching digits and enough token overlap.

In [ ]:
#| export
_DET = re.compile(r'^(the|a|an|this|that|these|those|its|their|our|your|his|her)\s+', re.I)
_WS  = re.compile(r'\s+')
_PRON = {'it','its','we','our','they','their','he','she','you','i','me','us','them','this','that',
         'these','those','which','who','what','there','here','one','ones','something','anything'}
_TOK = re.compile(r'[a-z0-9]+')
_NUM = re.compile(r'\d+')

`_norm` keeps any letter in any script. Spelled `[A-Za-z]` it returned None for Devanagari, and
a Devanagari corpus built an empty graph whatever its extraction did.

In [ ]:
#|export
def _norm(s):
    'Canonical surface form for a mention; None when the phrase is not entity-like.'
    if not s: return None
    s = _WS.sub(' ', s).strip().strip('.,;:!?()[]{}"\'`')
    s = _DET.sub('', s).strip()
    s = re.sub(r"'s$", '', s).strip()
    if not (2 <= len(s) <= 60): return None
    if len(s.split()) > 5: return None
    if s.lower() in _PRON: return None
    if not any(c.isalpha() for c in s): return None
    return s.lower()

@lru_cache(maxsize=1<<17)
def _toks(s):
    'Tokens for the lexical guard. UAX#29 treats `_` as a word joiner, so `fts_search` stays one'
    try: from apsw.unicode import word_iter, casefold
    except ImportError: return frozenset(_TOK.findall(s.lower()))
    return frozenset(casefold(t) for t in word_iter(s or '') if any(c.isalnum() for c in t))

@lru_cache(maxsize=1<<17)
def _acr(s):  return ''.join(w[0] for w in s.split() if w)

@lru_cache(maxsize=1<<17)
def _nums(s): return frozenset(_NUM.findall(s))

def _sentences(text):
    'UAX#29 sentence split via apsw; falls back to the whole text as one window.'
    try: from apsw.unicode import sentence_iter
    except ImportError: return [text]
    return [s for s in (x.strip() for x in sentence_iter(text)) if s]

def _jac(a, b):
    A, B = _toks(a), _toks(b)
    u = len(A) + len(B) - len(A & B)
    return len(A & B) / u if u else 0.0

Token overlap decides almost every pair and costs two cached frozensets. The digit and acronym
tests run after it, and can only veto or rescue what the tokens already ruled on.

In [ ]:
#|export
def _lex_ok(a, b, lex=0.34, cover=0.5):
    '''Lexical guard on a proposed merge.'''
    if a == b: return True
    A, B = _toks(a), _toks(b)
    ok = False
    if A and B:
        inter = len(A & B)
        ok = ((inter == min(len(A), len(B)) and inter/max(len(A), len(B)) >= cover)
              or inter/(len(A) + len(B) - inter) >= lex)
    if not ok and not (_acr(a) == b.lower() or _acr(b) == a.lower()): return False
    return _nums(a) == _nums(b)

The pairs that actually bite:

In [ ]:
cases = [('python 3.11','python 3.12',False), ('usearch','usearch index',True),
         ('hierarchical navigable small world','hnsw',True), ('polonium','isolated polonium',True),
         ('curie','marie curie isolated',False), ('marie curie','marie curie isolated',True),
         ('attention','multi head attention',False), ('wmt 2014 dataset','the wmt 2014 dataset',True)]
for a,b,want in cases:
    assert _lex_ok(a,b) == want, (a,b,want)
print(f'{len(cases)} guard cases pass')

`_toks` uses apsw's UAX#29 segmentation, which joins on `_`. Splitting `fts_search` into two
tokens would make it a subset of `search` and collapse the two by containment.

In [ ]:
import apsw, apsw.fts5
_c = apsw.Connection(':memory:'); apsw.fts5.register_tokenizers(_c, apsw.fts5.map_tokenizers)
for spec in (['porter','unicode61'], ['porter','simplify','casefold','1','unicodewords']):
    tk = _c.fts5_tokenizer(spec[0], spec[1:])
    got = [x[0] for x in tk(b'fts_search Running', apsw.FTS5_TOKENIZE_DOCUMENT, None, include_offsets=False)]
    print(f"{' '.join(spec):42s} -> {got}")

assert _toks('fts_search') == {'fts_search'}
assert not _lex_ok('fts_search', 'search'), 'identifiers must not collapse into their suffix'
print('\nidentifiers stay atomic')

## Code entities, from the AST

In [ ]:
#| export
_PY_SKIP = {'self','cls','super','print','len','str','int','float','bool','list','dict','set','tuple',
            'range','enumerate','zip','map','filter','isinstance','getattr','setattr','hasattr','type',
            'format','join','append','get','items','keys','values','open','sorted','sum','min','max'}

def _def_name(chunk):
    'Symbol defined by a code chunk, from pyparse metadata or the chunk source itself.'
    md = chunk.get('metadata') or {}
    if isinstance(md, dict) and md.get('name'): return md['name']
    try: tree = ast.parse(chunk['content'])
    except SyntaxError: return None
    n = first(tree.body, lambda x: isinstance(x, (ast.FunctionDef, ast.AsyncFunctionDef, ast.ClassDef)))
    return getattr(n, 'name', None)

def code_entities(chunk):
    'Exact symbols for a python chunk: (defined, called, imported). AST-derived, no model.'
    try: tree = ast.parse(chunk['content'])
    except (SyntaxError, ValueError): return None, L(), L()
    calls, imps = L(), L()
    for n in ast.walk(tree):
        if isinstance(n, ast.Call):
            f = n.func
            if   isinstance(f, ast.Name):      calls.append(f.id)
            elif isinstance(f, ast.Attribute): calls.append(f.attr)
        elif isinstance(n, ast.Import):
            imps += [a.name.split('.')[0] for a in n.names]
        elif isinstance(n, ast.ImportFrom):
            if n.module: imps.append(n.module.split('.')[0])
    keep = lambda s: s and s not in _PY_SKIP and not s.startswith('_') and len(s) > 2
    return _def_name(chunk), calls.filter(keep).unique(), imps.filter(keep).unique()

In [ ]:
# vruksha indexed against itself: the symbols come off the AST, so they are the real ones
from litesearch.data import dir2chunks
_chunks = dir2chunks('../vruksha', file_glob='*.py')
_ce = first(_chunks, lambda c: (c.get('metadata') or {}).get('name') == 'code_entities')
name, calls, imps = code_entities(_ce)
assert name == 'code_entities'
assert {'parse', 'walk'} <= set(calls)      # ast.parse and ast.walk, which it really does call

## Prose entities, from yake keyphrases

`prose_windows` groups entity surfaces one window per sentence. Page-sized windows make every
pair in a page a clique, and pruning does not recover from that.

In [ ]:
#| export
def _yake_terms(text, topk=12):
    'Keyphrases via yake — zero model, zero labels, and the default prose extractor.'
    try: from yake import KeywordExtractor
    except ImportError: return L()
    try: return L(KeywordExtractor(n=3, top=topk).extract_keywords(text)).map(lambda kv: kv[0])
    except Exception: return L()

def prose_windows(text,             # chunk text
                  topk=12,          # keyphrase count
                  terms_fn=None):   # (text, topk) -> terms; None -> yake
    'Entity surfaces grouped into co-occurrence windows — one per sentence.'
    terms = (terms_fn or _yake_terms)(text, topk)
    if not terms: return L()
    # the term list is fixed for the whole chunk, so it is lowercased once here rather than once
    # per sentence — `t.lower()` sat in the inner loop and ran topk times for every sentence
    low = [(t, t.lower()) for t in terms]
    wins = []
    for s in _sentences(text):
        sl = s.lower()
        hit = L([(t, 'keyphrase') for t, tl in low if tl in sl])
        if hit: wins.append(hit)
    return L(wins)

def text_entities(text, **kw):
    'Entity surfaces for a prose chunk, flattened across windows. Returns L of (surface, kind).'
    seen, out = set(), L()
    for w in prose_windows(text, **kw):
        for s, k in w:
            n = _norm(s)
            if n and n not in seen: seen.add(n); out.append((s, k))
    return out


In [ ]:
#| hide
# one window per sentence, and a surface the sentence already gave is not repeated
_t = 'Marie Curie studied polonium. Polonium decays by alpha emission.'
_w = prose_windows(_t, terms_fn=lambda t, k: L(['polonium', 'marie curie']))
assert len(_w) == 2, _w
for _win in _w: assert len(_win) == len({s.lower() for s, _ in _win}), _win
assert all(k == 'keyphrase' for win in _w for _, k in win)

# terms_fn replaces yake rather than adding to it
assert prose_windows(_t, terms_fn=lambda t, k: L()) == L()
assert [s for s, _ in text_entities(_t, terms_fn=lambda t, k: L(['polonium']))] == ['polonium']

# scripts other than Latin are entities too — the ASCII guard used to drop all of them
assert _norm('धर्मक्षेत्रे') == 'धर्मक्षेत्रे'
assert _norm('Кюри') and _norm('居里')
assert _norm('123') is None and _norm('...') is None


In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()